# B1 — Unconstrained PPO

**Purpose:** Train unconstrained PPO with CjMmCriterion on training days (Jan 1–6);
evaluate on the held-out test set (Jan 7–29).

**Design:**
- Training environment: 6 days concatenated into a single long episode (~4.3M steps).
- Reward: `CjMmCriterion(φ=0.01)` — PnL minus inventory-aversion term.
- Observations normalised via `VecNormalize` (running stats computed during training).
- Test evaluation: per-day, frozen VecNormalize stats from training.

**Outputs:**
- `models/ppo_b1_doge.zip` — trained PPO weights
- `models/vecnorm_b1.pkl` — VecNormalize running statistics
- `results/b1_test_results.csv` — per-day test metrics

In [ ]:
import sys
import pathlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

PROJECT_ROOT = pathlib.Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from stable_baselines3 import PPO

from procs.gym.experiment_config import ReplayExperimentConfig
from procs.gym.data_loader import load_multi_day
from procs.gym.sb3_wrapper import StableBaselinesTradingEnvironment
from procs.gym.notebook_support import (
    build_replay_env,
    make_vecnorm,
    evaluate_rl_per_day,
)
from procs.rewards import CjMmCriterion

cfg = ReplayExperimentConfig()
cfg.ensure_artifact_dirs()
print(f"Repo root : {cfg.repo_root}")
print(f"Models    : {cfg.models_dir}")
print(f"Results   : {cfg.results_dir}")

## Section 1 — Data Loading and Train/Test Split

In [ ]:
TRAIN_DAYS = 6

daily_S, daily_dt, dates = load_multi_day(str(cfg.datasets_dir), pair=cfg.pair)

train_S     = daily_S[:TRAIN_DAYS]
train_dt    = daily_dt[:TRAIN_DAYS]
train_dates = dates[:TRAIN_DAYS]

test_S      = daily_S[TRAIN_DAYS:]
test_dt     = daily_dt[TRAIN_DAYS:]
test_dates  = dates[TRAIN_DAYS:]

# Concatenate training days into a single long array
S_train  = np.concatenate(train_S)
dt_train = np.concatenate(train_dt)

print(f"Training days : {len(train_S)}  ({train_dates[0]} → {train_dates[-1]})")
print(f"Test days     : {len(test_S)}  ({test_dates[0]} → {test_dates[-1]})")
print(f"S_train length: {len(S_train):,} snapshots (~{len(S_train)/1e6:.2f}M)")

## Section 2 — Build Multi-Day Training Environment

In [ ]:
reward_fn_train = CjMmCriterion(per_step_inventory_aversion=cfg.phi)

train_env  = build_replay_env(S_train, dt_train, cfg, reward_fn=reward_fn_train)
train_sb3  = StableBaselinesTradingEnvironment(train_env)
train_vn   = make_vecnorm(train_sb3, cfg, training=True, norm_reward=True)

obs_dim = train_env.observation_space.shape[0]
print(f"Observation dim : {obs_dim}")
print(f"Action dim      : {train_env.action_space.shape[0]}")
print(f"Episode length  : {len(S_train):,} steps")

## Section 3 — PPO Training

**`TOTAL_TIMESTEPS` guide:**
- Quick local test: `200_000`
- Full Snellius run: `max(2 * len(S_train), 2_000_000)`

In [ ]:
TOTAL_TIMESTEPS = max(len(S_train), 1_000_000)  # ≥1 full pass or 1M steps
print(f"TOTAL_TIMESTEPS = {TOTAL_TIMESTEPS:,}  ({TOTAL_TIMESTEPS/len(S_train):.2f}× data)")

model_b1 = PPO(
    "MlpPolicy",
    train_vn,
    **cfg.ppo_kwargs(),
    tensorboard_log=str(cfg.repo_root / "tb_logs" / "b1"),
    verbose=1,
    device="cpu",
)

print(f"PPO kwargs: {cfg.ppo_kwargs()}")
model_b1.learn(total_timesteps=TOTAL_TIMESTEPS)

In [ ]:
model_b1.save(str(cfg.model_path("ppo_b1_doge")))
train_vn.save(str(cfg.vecnorm_path("vecnorm_b1")))
print(f"Saved model  → {cfg.model_path('ppo_b1_doge')}.zip")
print(f"Saved vecnorm → {cfg.vecnorm_path('vecnorm_b1')}")

## Section 4 — Training Diagnostics

Quick diagnostic from the last logged progress (PPO prints every `n_steps` rollout).
For full curves, run: `tensorboard --logdir tb_logs/b1`

In [ ]:
# Parse TensorBoard event file for explained variance and entropy
try:
    from tensorflow.python.summary.summary_iterator import summary_iterator
    tb_dir = cfg.repo_root / "tb_logs" / "b1"
    ev_files = sorted(tb_dir.glob("events.out.tfevents.*"))
    if ev_files:
        ev_data = {"step": [], "explained_variance": [], "entropy_loss": []}
        for event in summary_iterator(str(ev_files[-1])):
            for v in event.summary.value:
                if v.tag == "train/explained_variance":
                    ev_data["step"].append(event.step)
                    ev_data["explained_variance"].append(v.simple_value)
                elif v.tag == "train/entropy_loss":
                    ev_data["entropy_loss"].append(v.simple_value)

        df_tb = pd.DataFrame(ev_data).dropna()
        if len(df_tb):
            fig, axes = plt.subplots(1, 2, figsize=(12, 4))
            axes[0].plot(df_tb["step"], df_tb["explained_variance"])
            axes[0].axhline(0.5, color="red", linestyle="--", alpha=0.5, label="target ≥ 0.5")
            axes[0].set_title("Explained Variance")
            axes[0].legend(); axes[0].grid(True, alpha=0.3)
            axes[1].plot(df_tb["step"], df_tb.get("entropy_loss", []))
            axes[1].set_title("Entropy Loss")
            axes[1].grid(True, alpha=0.3)
            plt.suptitle("B1 Training Diagnostics")
            plt.tight_layout()
            plt.show()
        print(f"Final explained variance: {df_tb['explained_variance'].iloc[-1]:.4f}")
    else:
        print("No TensorBoard event files found — skipping diagnostic plot.")
except Exception as e:
    print(f"TensorBoard parsing unavailable ({e}). Run: tensorboard --logdir tb_logs/b1")

## Section 5 — Test Evaluation (Days 7–29)

In [ ]:
model_b1_loaded = PPO.load(str(cfg.model_path("ppo_b1_doge")), device="cpu")

df_b1 = evaluate_rl_per_day(
    model=model_b1_loaded,
    vecnorm_path=cfg.vecnorm_path("vecnorm_b1"),
    test_S=test_S,
    test_dt=test_dt,
    test_dates=test_dates,
    config=cfg,
    seed=42,
)

print(f"Evaluated {len(df_b1)} test days.")

## Section 6 — Results Display and Save

In [ ]:
pd.set_option("display.float_format", "{:.4f}".format)
METRICS = ["Sharpe", "Sortino", "Max DD", "P&L-to-MAP", "Final PnL"]

summary = pd.DataFrame({
    "Mean":   df_b1[METRICS].mean(),
    "Std":    df_b1[METRICS].std(),
    "Median": df_b1[METRICS].median(),
}).T

print("=== B1 Per-day Test Results ===")
print(df_b1[METRICS].to_string())
print("\n=== B1 Summary ===")
print(summary.to_string())

In [ ]:
out_path = cfg.result_path("b1_test_results.csv")
df_b1.to_csv(out_path)
print(f"Saved → {out_path}")